# GPR — Plotting

Loads artifacts produced by `GPR_calculation.ipynb` from `OUTPUT_FOLDER` and produces every figure. Tune any cell freely per sample size — the underlying fit is not re-run.


In [ ]:
import  os
import  json
import  joblib
import  numpy as np
import  pandas as pd
import  thermoift.PLOT_SETTINGS as ps
from    thermoift import MLPostprocessing, MLPreprocessing
from    thermoift import plot_correlation_heatmap


In [ ]:
OUTPUT_FOLDER = "GPR_RESIDUAL_OUTPUTS"


## Load calculation artifacts


In [ ]:
meta = json.load(open(os.path.join(OUTPUT_FOLDER, "GPR_artifacts_meta.json")))
features = meta["features"]
target   = meta["target"]

gpr_model = joblib.load(os.path.join(OUTPUT_FOLDER, "GPR_residual_model.joblib"))
df        = pd.read_parquet(os.path.join(OUTPUT_FOLDER, "GPR_df_full.parquet"))

_splits = np.load(os.path.join(OUTPUT_FOLDER, "GPR_splits.npz"))
train_idx = _splits["train_idx"]
test_idx  = _splits["test_idx"]
val_idx   = _splits["val_idx"]

X_train = df.loc[train_idx, features]
X_test  = df.loc[test_idx,  features]
X_val   = df.loc[val_idx,   features]
y_train = df.loc[train_idx, target]
y_test  = df.loc[test_idx,  target]
y_val   = df.loc[val_idx,   target]

_preds = np.load(os.path.join(OUTPUT_FOLDER, "GPR_predictions.npz"))
y_train_pred, y_train_std = _preds["y_train_pred"], _preds["y_train_std"]
y_test_pred,  y_test_std  = _preds["y_test_pred"],  _preds["y_test_std"]
y_val_pred,   y_val_std   = _preds["y_val_pred"],   _preds["y_val_std"]

_recon = np.load(os.path.join(OUTPUT_FOLDER, "GPR_gamma_reconstructed.npz"))
gamma_cDFT_train = _recon["gamma_cDFT_train"]; gamma_pred_train = _recon["gamma_pred_train"]
gamma_cDFT_test  = _recon["gamma_cDFT_test"];  gamma_pred_test  = _recon["gamma_pred_test"]
gamma_cDFT_val   = _recon["gamma_cDFT_val"];   gamma_pred_val   = _recon["gamma_pred_val"]

_feature_importances = np.load(os.path.join(OUTPUT_FOLDER, "GPR_feature_importances.npy"))

print(f"Loaded artifacts from: {OUTPUT_FOLDER}")
print(f"Features: {features}")
print(f"train / test / val sizes: {len(train_idx)} / {len(test_idx)} / {len(val_idx)}")


In [ ]:
post = MLPostprocessing(
    y_true              = y_test,
    y_pred              = y_test_pred,
    target              = "delta_gamma",
    feature_importances = _feature_importances,
    feature_names       = features,
    datasets            = {
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)


## EDA on residual


In [ ]:
plot_correlation_heatmap(df, features, target, save_path="GPR_residual_correlation", folder=OUTPUT_FOLDER)

In [ ]:
prep = MLPreprocessing(df=df, features=features, target=target)

prep.plot_scatter(
    x           = "T",
    y           = target,
    color_by    = "P",
    hline       = 0.0,
    save_path   = "GPR_residual_vs_T",
    folder      = OUTPUT_FOLDER,
)

prep.plot_scatter(
    x           = "P",
    y           = target,
    color_by    = "T",
    hline       = 0.0,
    save_path   = "GPR_residual_vs_P",
    folder      = OUTPUT_FOLDER,
)

## Residual-model diagnostics


In [ ]:
# Parity plot
post.plot_parity(model_name="GPR (RBF)", save_path="GPR_gamma_parity_plot", folder=OUTPUT_FOLDER)

In [ ]:
# Parity + residual distribution (combined two-panel figure)
post.plot_parity_with_residual_distribution(
    n_bins      = 30,
    save_path   = "GPR_parity_residual_distribution",
    folder      = OUTPUT_FOLDER,
)

In [ ]:
# Residual distribution
post.plot_residual_distribution(save_path="GPR_gamma_residual_distribution", folder=OUTPUT_FOLDER)

In [ ]:
# Residuals vs Predicted
post.plot_residual_vs_predicted(save_path="GPR_gamma_residual_vs_predicted", folder=OUTPUT_FOLDER, 
                                y_range= (-0.5, 0.5))

In [ ]:
# GPR: Predictive std histogram
post.plot_std_histogram(y_test_std, save_path="GPR_gamma_std_histogram", folder=OUTPUT_FOLDER)

In [ ]:
# GPR: |error| vs predictive std (calibration)
post.plot_error_vs_std(y_test_std, save_path="GPR_gamma_error_vs_std", folder=OUTPUT_FOLDER)

## Response curves & 2D surface


In [ ]:
# GPR: Response curves — saved individually per feature
X_ref = X_train.median().values
post.plot_response_curves(
    model               = gpr_model,
    X_ref               = X_ref,
    feature_names       = features,
    X_train             = X_train,
    n_points            = 100,
    return_std          = True,
    save_individually   = True,
    save_path           = "GPR_gamma_response",
    folder              = OUTPUT_FOLDER,
)

In [ ]:
# 2D Response Surface: T vs P (all z_* fixed at training median)
post.plot_2d_response_surface(
    model         = gpr_model,
    X_train       = X_train,
    y_train       = y_train,
    feature_names = features,
    T_name        = "T",
    P_name        = "P",
    n_grid        = 80,
    return_std    = False,
    save_path     = "GPR_2D_response_T_P",
    folder        = OUTPUT_FOLDER,
)

## Reconstructed γ_cDFT parity


In [ ]:
# Parity plot: gamma_cDFT (actual) vs gamma_pred (reconstructed) with ±2σ uncertainty
post.plot_reconstructed_parity(
    datasets_reconstructed={
        "train": (gamma_cDFT_train, gamma_pred_train, y_train_std),
        "test":  (gamma_cDFT_test,  gamma_pred_test,  y_test_std),
        "val":   (gamma_cDFT_val,   gamma_pred_val,   y_val_std),
    },
    n_sigma=2.0,
    model_name="GPR",
    save_path="GPR_gamma_reconstructed_parity",
    folder=OUTPUT_FOLDER,
)

In [ ]:
post.print_summary()
